# Day 3-02｜YOLO Tracking 與 ByteTrack ID


## 執行階段提醒

- 這一單元直接讓 `YOLO + ByteTrack` 在整段影片上跑 tracking。
- 跟前一單元不同，這裡不再手動配對兩張圖，而是讓 tracker 自動維持 `track_id`。


## 課程流程

1. 設定影片、模型與 tracking 參數。
2. 產生 ByteTrack preview 影片。
3. 檢查 `track_id`、bbox、confidence 的結構化輸出。
4. 彙整每個 `track_id` 出現了幾個 frame。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


## Step 1｜設定影片、模型與 tracking 參數


In [ ]:
import pandas as pd

from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    detector_model_path,
    first_reference_video,
    write_bytetrack_preview_video,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
OUTPUT_PATH = COURSE_ROOT / "assets" / "results" / "d3_02_bytetrack_preview.mp4"
START_FRAME = 0
MAX_FRAMES = 120
CONF = 0.25
IMGSZ = 960

print("video:", VIDEO_PATH)
print("model:", MODEL_PATH)
print("output:", OUTPUT_PATH)
print("start frame:", START_FRAME)
print("max frames:", MAX_FRAMES)


## Step 2｜執行 ByteTrack，輸出 preview 影片


In [ ]:
preview_video, records = write_bytetrack_preview_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=OUTPUT_PATH,
    max_frames=MAX_FRAMES,
    conf=CONF,
    imgsz=IMGSZ,
    start_frame=START_FRAME,
)
preview_json = preview_video.with_suffix(".json")

print("preview video:", preview_video)
print("preview json:", preview_json)
print("records:", len(records))
display_video_in_notebook(preview_video, loop=True)


## Step 3｜檢查每筆 tracking record


In [ ]:
track_columns = ["frame", "track_id", "class_id", "class_name", "confidence", "bbox_xyxy"]
tracks = pd.DataFrame(records, columns=track_columns)
tracks.head(10)


`track_id` 是後續 BEV 路徑與戰術資料表的主鍵。若 detector 漏偵、重複框過多或遮擋時間過長，ID 仍可能中斷或交換。


## Step 4｜統計每個 track_id 持續了多少 frame


In [ ]:
if tracks.empty:
    track_summary = pd.DataFrame(columns=["track_id", "frames_seen", "first_frame", "last_frame", "mean_confidence"])
else:
    track_summary = (
        tracks.groupby("track_id", dropna=False)
        .agg(
            frames_seen=("frame", "count"),
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
            mean_confidence=("confidence", "mean"),
        )
        .sort_values(["frames_seen", "first_frame"], ascending=[False, True])
        .reset_index()
    )
track_summary.head(15)


## 本單元產出檔案

- `assets/results/d3_02_bytetrack_preview.mp4`
- `assets/results/d3_02_bytetrack_preview.json`
